In [33]:

import dataclasses as dc
from pathlib import Path
import re

iso_slug_regex = re.compile(r'(\d+)([A-Z][a-zA-Z]*)(\d*)')
atom_num_regex = re.compile(r'\((\d+)([A-Z][a-zA-Z]*)\)(\d*)')

def iso_slug_to_iso_name(iso_slug):
	iso_name = ''
	for match in iso_slug_regex.finditer(iso_slug):
		iso_mass = int(match[1])
		iso_atom = match[2]
		iso_multiplicity = match[3]
		iso_name += f'({iso_mass}{iso_atom}){iso_multiplicity}'
	return iso_name

def iso_name_to_iso_spec(iso_name):
	iso_spec = []
	for match in atom_num_regex.finditer(iso_name):
		iso_spec.append(
			((int(match[1]), match[2]), int(match[3]) if len(match[3]) > 0 else 1)
		)
	return tuple(iso_spec)

def mol_spec_from_iso_name(iso_name):
	mol_spec : dict[str,int] = dict()
	for match in atom_num_regex.finditer(iso_name):
		atom_name = match[2]
		atom_multiplicity = int(match[3]) if len(match[3]) > 0 else 1
		mol_spec[atom_name] = mol_spec.get(atom_name, 0) + atom_multiplicity
	
	for k in tuple(mol_spec.keys()):
		if mol_spec[k] == 0:
			del mol_spec
	
	return tuple(sorted((k,mol_spec[k]) for k in sorted(mol_spec.keys())))

def parse_pc_data_file_name(fname : str):
	iso_slug, remainder = fname.split('__', 1)
	remainder, ftype = remainder.rsplit('.', 1)
	dsname, t_cont = remainder.rsplit('_T', 1)
	
	iso_name = iso_slug_to_iso_name(iso_slug)
	
	return (
		ftype,
		iso_name,
		mol_spec_from_iso_name(iso_name),
		dsname,
		t_cont
	)

@dc.dataclass
class PCDataFileSet:
	t_cont : float
	mol_spec : tuple
	iso_name : str
	contbins : None | Path = None
	continuum : None | Path = None
	stronglines : None | Path = None
	

pc_data_files = dict()

for pc_data_file in Path('./test_data/EXOMOL_test').iterdir():
	ftype, iso_name, mol_spec, dsname, t_cont = parse_pc_data_file_name(pc_data_file.name)
	
	setattr(
		pc_data_files
			.setdefault(mol_spec, dict())
				.setdefault(iso_name,dict())
					.setdefault(dsname,dict())
						.setdefault(
							float(t_cont),
							PCDataFileSet(
								t_cont=float(t_cont),
								mol_spec = mol_spec,
								iso_name = iso_name,
							)
						),
		ftype,
		pc_data_file
	)


for mol_spec,a in pc_data_files.items():
	print(f'{mol_spec}')
	for iso_name, b in a.items():
		print(f'\t{iso_name}')
		for dsname, c in b.items():
			print(f'\t\t{dsname}')
			for t_cont, d in c.items():
				print(f'\t\t\t{t_cont}\n\t\t\t\t{d}')







(('C', 1), ('H', 4))
	(12C)(1H)4
		YT34to10
			250.0
				PCDataFileSet(t_cont=250.0, mol_spec=(('C', 1), ('H', 4)), iso_name='(12C)(1H)4', contbins=PosixPath('test_data/EXOMOL_test/12C-1H4__YT34to10_T250.0.contbins'), continuum=PosixPath('test_data/EXOMOL_test/12C-1H4__YT34to10_T250.0.continuum'), stronglines=PosixPath('test_data/EXOMOL_test/12C-1H4__YT34to10_T250.0.stronglines'))
			296.0
				PCDataFileSet(t_cont=296.0, mol_spec=(('C', 1), ('H', 4)), iso_name='(12C)(1H)4', contbins=PosixPath('test_data/EXOMOL_test/12C-1H4__YT34to10_T296.0.contbins'), continuum=PosixPath('test_data/EXOMOL_test/12C-1H4__YT34to10_T296.0.continuum'), stronglines=PosixPath('test_data/EXOMOL_test/12C-1H4__YT34to10_T296.0.stronglines'))
			200.0
				PCDataFileSet(t_cont=200.0, mol_spec=(('C', 1), ('H', 4)), iso_name='(12C)(1H)4', contbins=PosixPath('test_data/EXOMOL_test/12C-1H4__YT34to10_T200.0.contbins'), continuum=PosixPath('test_data/EXOMOL_test/12C-1H4__YT34to10_T200.0.continuum'), stronglines=PosixPath

In [6]:
from archnemesis.database.filetypes.ans_pseudo_continuum_file import AnsPseudoContinuumFile


ans_pc_file = AnsPseudoContinuumFile(Path('./test_data/test_pc.h5'))

In [ ]:
from typing import Self

import numpy as np

class DtypeStringParser:
	def __init__(self):
		self.TOK_SEP = ';'
		self.TOK_EOF = 'END_OF_FILE'
		
		self.reset()
		
	
	def reset(self, s : str | None = None) -> Self:
		self.pos = 0
		self.end = len(s) if s is not None else None
		self.current_token = None
		self.exhausted = False
		return self
		
	def next_token(self, s) -> str:
		idx = s.find(self.TOK_SEP, self.pos, self.end)
		
		if idx > 0:
			self.current_token = s[self.pos:idx]
			self.pos = idx + len(self.TOK_SEP)
		else:
			if not self.exhausted:
				self.pos = len(s)
				self.current_token = '}'
				self.exhausted = True
			else:
				self.current_token = self.TOK_EOF
		
		return self.current_token
	
	def is_exhausted(self, s) -> bool:
		return self.exhausted
	
	def parse_structured(self, s : str) -> np.dtype:
		"""
		Parse a string for a structured dtype
		"""
		fnames = []
		foffsets = []
		fdtypes = []
		
		while (tok := self.next_token(s)) != '}':
			#print(f'110 :: {tok}')
			fnames.append(tok)
			
			tok = self.next_token(s)
			#print(f'120 :: {tok}')
			foffsets.append(int(tok))
			
			fdtype = self.parse(s)
			fdtypes.append(fdtype)
		
		result = np.dtype({'names':fnames, 'formats' : fdtypes, 'offsets':foffsets})
		
		#print(f'130 :: {self.current_token}')
		assert self.current_token == '}', f'Expected token "}}", but got token "{tok}"'
		return result
	
	def parse_array(self, s : str) -> np.dtype:
		"""
		Parse a string for an array dtype
		"""
		
		tok = self.next_token(s)
		#print(f'210 :: {tok}')
		shape = tuple(int(x.strip()) for x in tok[1:-1].split(','))
		
		fdtype = self.parse(s)
		
		result = np.dtype((fdtype,shape))
		
		tok = self.next_token(s)
		#print(f'220 :: {tok}')
		assert tok == '}', f'Expected token "}}", but got token "{tok}"'
		return result
	
	def parse_type(self, s : str) -> np.dtype:
		"""
		Parse a string for a simple dtype
		"""
		tok = self.next_token(s)
		#print(f'310 :: {tok}')
		result = np.dtype(tok)
		
		tok = self.next_token(s)
		#print(f'320 :: {tok}')
		assert tok == '}', f'Expected token "}}", but got token "{tok}"'
		return result
	
	def parse(self, s : str) -> np.dtype:
		"""
		Parse a string that specifies a general dtype
		"""
		tok = self.next_token(s)
		
		result = None
		
		if tok == 'STRUCTURED{':
			#print(f'100 :: {tok}')
			result = self.parse_structured(s)
		elif tok == 'ARRAY{':
			#print(f'200 :: {tok}')
			result = self.parse_array(s)
		elif tok == 'TYPE{':
			#print(f'300 :: {tok}')
			result = self.parse_type(s)
		elif tok == '}':
			raise RuntimeError('Unexpected end of section')
		elif tok == self.TOK_EOF:
			raise RuntimeError('Unexpected end of input')
		else:
			raise RuntimeError(f'Unknown Token "{tok}"')
		
		return result


_dtype_string_parser = DtypeStringParser()

def dtype_from_string(string : str) -> np.dtype:
	return _dtype_string_parser.reset().parse(string)

In [27]:



for a in pc_data_files.values():
	for b in a.values():
		for c in b.values():
			pc_dfs = c[250.0]
			break
		break
	break
		

print(pc_dfs)

cont_bin_edge = np.fromfile(pc_dfs.contbins)
#print(f'{cont_bin_edge=}')
cont_bin_center = 0.5*(cont_bin_edge[:-1] +cont_bin_edge[1:])
#print(f'{cont_bin_center=}')
cont_bin_width = np.diff(cont_bin_edge)
#print(f'{cont_bin_width=}')


with open(pc_dfs.continuum, 'rb') as f:
	hdr = b''
	while len(hdr) < 1024 and (len(hdr) == 0 or hdr[-1] != 0):
		hdr += f.read(4)
	
	print(f'{hdr=}')

	continuum_dtype = dtype_from_string(hdr.decode('ascii'))
	print(f'{continuum_dtype=}')
	
	cont_data = np.fromfile(f, dtype=continuum_dtype)
	print(f'{cont_data[:5]=}')

with open(pc_dfs.stronglines, 'rb') as f:
	hdr = b''
	while len(hdr) < 1024 and (len(hdr) == 0 or hdr[-1] != 0):
		hdr += f.read(4)
	
	print(f'{hdr=}')

	stronglines_dtype = dtype_from_string(hdr.decode('ascii'))
	print(f'{stronglines_dtype=}')
	
	stronglines_data = np.fromfile(f, dtype=stronglines_dtype)
	print(f'{stronglines_data[:5]=}')

PCDataFileSet(t_cont=250.0, contbins=PosixPath('test_data/EXOMOL_test/12C-1H4__YT34to10_T250.0.contbins'), continuum=PosixPath('test_data/EXOMOL_test/12C-1H4__YT34to10_T250.0.continuum'), stronglines=PosixPath('test_data/EXOMOL_test/12C-1H4__YT34to10_T250.0.stronglines'))
hdr=b'STRUCTURED{;line_strength_sum;0;TYPE{;<f8;};strength_weighted_sum_E";8;TYPE{;<f8;};strength_weighted_gamma_self;16;TYPE{;<f8;};strength_weighted_n_self;24;TYPE{;<f8;};strength_weighted_gamma_air;32;TYPE{;<f8;};strength_weighted_n_air;40;TYPE{;<f8;};strength_weighted_gamma_H2;48;TYPE{;<f8;};strength_weighted_n_H2;56;TYPE{;<f8;};strength_weighted_gamma_He;64;TYPE{;<f8;};strength_weighted_n_He;72;TYPE{;<f8;};strength_weighted_gamma_CO2;80;TYPE{;<f8;};strength_weighted_n_CO2;88;TYPE{;<f8;};}\x00\x00\x00\x00'
continuum_dtype=dtype([('line_strength_sum', '<f8'), ('strength_weighted_sum_E"', '<f8'), ('strength_weighted_gamma_self', '<f8'), ('strength_weighted_n_self', '<f8'), ('strength_weighted_gamma_air', '<f8'), ('s

In [ ]:
from archnemesis.database.data_holders.pseudo_continuum_data_holder import PseudoContinuumDataHolder
from archnemesis.database.data_holders.pseudo_continuum_broadener_part import PseudoContinuumBroadenerPart

from archnemesis.Data.gas_data import gas_info, atom_mass

atom_names = tuple(atom_mass.keys())

def mol_name_to_mol_spec(mol_name, atom_names = atom_names):
	mol_spec = dict()
	
	while len(mol_name) > 0:
		atom_name_found = False
		for atom_name in atom_names:
			
			if mol_name.startswith(atom_name):
				atom_name_found = True
				atom_multiplicity = 1
				mol_name = mol_name[len(atom_name):]
				i = 0
				while mol_name[i].isdigit():
					i+=1
				if i > 0:
					atom_multiplicity = int(mol_name[:i])
					mol_name = mol_name[i:]
				mol_spec[atom_name] = mol_spec.get(atom_name, 0) + atom_multiplicity
	
	return tuple(sorted((k,v) for k,v in mol_spec.items()))
	
	

rt_mol_id = None
rt_iso_id = None

uncontained_iso_atom_regex = re.compile(r'[A-Z][a-zA-Z]*(?!\))')
uncontained_iso_atom_subs = {'H' : '(1H)', 'D' : '(2H)'}

for mol_id, mol_data in gas_info.items():
	mol_name = mol_data['name']
	
	rt_mol_spec = mol_name_to_mol_spec(mol_name, atom_names=atom_names)
	if rt_mol_spec != pc_dfs.mol_spec:
		continue
	
	rt_mol_id = mol_id
	
	for iso_id, iso_data in mol_data['isotope']:
		iso_name = iso_data['name']
		canonical_iso_name = ''
		i = 0
		j = 0
		for match in uncontained_iso_atom_regex.finditer(iso_name):
			j = match.start
			canonical_iso_name += iso_name[i:j]
			canonical_iso_name += uncontained_iso_atom_subs(match[0])
			i = match.end
		
		if canonical_iso_name != pc_dfs.iso_name:
			continue
		rt_iso_id = iso_id
		

if rt_mol_id is None:
	raise RuntimeError(f'Could not fine RADTRAN molecule id for {pc_dfs.mol_spec}')
if rt_iso_id is None:
	raise RuntimeError(f'Could not fine RADTRAN local iso id for {pc_dfs.iso_name}')

pc_dh = PseudoContinuumDataHolder(
	"EXOMOL_test",
	"This data is a test that has been extracted from exomol",
	
	t_cont = pc_dfs.t_cont,
	
	mol_id = rt_mol_id,
	local_iso_id = rt_iso_id,
)



PatternError: look-behind requires fixed-width pattern